# 04 · Retrieval Metrics — Precision@K, Recall@K, MRR
How do you know retrieval is good? Measure it against labeled queries. Every metric here is computed from scratch so you can check it by hand.

In [ ]:
# A REAL but simple vectorizer: bag-of-words (term frequency). Offline, deterministic,
# and good enough that similar documents (sharing words) get similar vectors — so
# cosine similarity and retrieval metrics are genuinely meaningful and hand-checkable.
#
# NOTE: production RAG uses NEURAL embeddings (e.g. Jina, OpenAI, sentence-transformers)
# that also capture SYNONYMS ("car" ~ "automobile") with no shared words. The MATH below
# (cosine, retrieval, metrics) is identical; only the vectors get smarter. Where a cell
# says "swap in a real embedder", that's the one line that changes.
import numpy as np, re
def tokenize(text): return re.findall(r"[a-z0-9]+", text.lower())
def build_vocab(texts):
    vocab={}
    for t in texts:
        for w in tokenize(t):
            if w not in vocab: vocab[w]=len(vocab)
    return vocab
def vectorize(text, vocab):
    v=np.zeros(len(vocab))
    for w in tokenize(text):
        if w in vocab: v[vocab[w]]+=1.0
    return v
def cosine(a,b):
    na,nb=np.linalg.norm(a),np.linalg.norm(b)
    return float(a@b/(na*nb)) if na and nb else 0.0

## 1. Load corpus + labeled queries (the ground truth)

In [ ]:
import pandas as pd
docs = pd.read_csv("../dataset/corpus.csv")
queries = pd.read_csv("../dataset/queries.csv")
queries["relevant_ids"] = queries["relevant_ids"].apply(lambda s: set(s.split()))
vocab = build_vocab(docs["text"].tolist())
doc_vecs = [vectorize(t, vocab) for t in docs["text"]]
id_list = docs["id"].tolist()

def rank(query):
    qv = vectorize(query, vocab)
    order = sorted(range(len(docs)), key=lambda i: -cosine(qv, doc_vecs[i]))
    return [id_list[i] for i in order]
print(f"{len(docs)} docs, {len(queries)} labeled queries")

## 2. The three metrics, from scratch

In [ ]:
def precision_at_k(ranked, relevant, k):
    topk = ranked[:k]
    return sum(1 for d in topk if d in relevant) / k

def recall_at_k(ranked, relevant, k):
    if not relevant: return 0.0
    topk = ranked[:k]
    return sum(1 for d in topk if d in relevant) / len(relevant)

def reciprocal_rank(ranked, relevant):
    for i, d in enumerate(ranked, 1):
        if d in relevant: return 1.0/i
    return 0.0
print("metrics defined")

## 3. Score every query

In [ ]:
Ks=[1,3,5]; rows=[]
for _,q in queries.iterrows():
    r=rank(q["query"]); rel=q["relevant_ids"]
    row={"qid":q["qid"],"n_rel":len(rel),"RR":round(reciprocal_rank(r,rel),3)}
    for k in Ks:
        row[f"P@{k}"]=round(precision_at_k(r,rel,k),3)
        row[f"R@{k}"]=round(recall_at_k(r,rel,k),3)
    rows.append(row)
res=pd.DataFrame(rows); res

## 4. Aggregate: MRR and mean P@K / R@K

In [ ]:
print("MRR =", round(res["RR"].mean(),3))
for k in Ks:
    print(f"  mean P@{k} = {res[f'P@{k}'].mean():.3f}   mean R@{k} = {res[f'R@{k}'].mean():.3f}")

**Read it:**
```
  Precision@K = (relevant in top K) / K          -> falls as K grows
  Recall@K    = (relevant in top K) / (all relevant) -> rises as K grows
  RR          = 1 / (rank of first relevant hit)
  MRR         = average RR over all queries
```
Pick a single query row and verify a number by hand against these formulas — that's the point of learning on a tiny labeled set. (This is the same math as your `retrieval_metrics` module.)